In [1]:
#!/usr/bin/env python3
# coding: utf-8

import os
import json
from typing import Any, Dict
import firebase_admin
from firebase_admin import credentials, firestore
import pandas as pd

# -------- CONFIGURAÇÃO --------
CREDENTIALS_PATH = '../../private_key.json'        # caminho para a sua chave do service account
COLLECTION_NAME = 'patrimonios_santos'        # coleção a exportar
OUTPUT_CSV = f'{COLLECTION_NAME}_export.csv'
OUTPUT_XLSX = f'{COLLECTION_NAME}_export.xlsx'
# --------------------------------

def ensure_firebase_initialized():
    if not firebase_admin._apps:
        cred = credentials.Certificate(CREDENTIALS_PATH)
        firebase_admin.initialize_app(cred)

def flatten_doc(doc: Dict[str, Any], parent_key: str = '', sep: str = '.') -> Dict[str, Any]:
    """
    Flatten nested dicts one level deep (recursive).
    Lists are kept as lists (handled later).
    """
    items = {}
    for k, v in doc.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            # recursion for nested dicts
            items.update(flatten_doc(v, parent_key=new_key, sep=sep))
        else:
            items[new_key] = v
    return items

def normalize_value(value):
    """
    Convert common Firestore types to plain python values for DataFrame.
    - lists -> join with ";;" (so commas in urls don't break CSV)
    - None stays None
    - other types returned as-is
    """
    if isinstance(value, list):
        # convert nested dicts in list to JSON string, else join primitives
        safe_items = []
        for it in value:
            if isinstance(it, dict):
                safe_items.append(json.dumps(it, ensure_ascii=False))
            else:
                safe_items.append(str(it))
        return ';;'.join(safe_items)
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)
    return value

def export_collection_to_dataframe(collection_name: str) -> pd.DataFrame:
    ensure_firebase_initialized()
    db = firestore.client()

    docs = list(db.collection(collection_name).stream())
    rows = []
    for d in docs:
        data = d.to_dict() or {}
        # include document id as a column
        data['_doc_id'] = d.id

        # flatten nested dicts
        flat = flatten_doc(data)

        # normalize values (lists -> joined string, dicts -> json string)
        flat_normalized = {k: normalize_value(v) for k, v in flat.items()}

        rows.append(flat_normalized)

    if not rows:
        print("Coleção vazia ou nenhum documento encontrado.")
        return pd.DataFrame()

    # create DataFrame — pandas preencherá NaN quando chaves faltarem
    df = pd.DataFrame(rows)

    # reordenar: colocar _doc_id como primeira coluna (se existir)
    if '_doc_id' in df.columns:
        cols = df.columns.tolist()
        cols.insert(0, cols.pop(cols.index('_doc_id')))
        df = df[cols]

    return df

def main():
    print(f'Exportando coleção "{COLLECTION_NAME}" para DataFrame...')
    df = export_collection_to_dataframe(COLLECTION_NAME)

    if df.empty:
        print("DataFrame vazio. Nenhum arquivo gerado.")
        return

    # salvar CSV e Excel
    print(f'Número de documentos: {len(df)}')
    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    print(f'CSV salvo: {OUTPUT_CSV}')
    df.to_excel(OUTPUT_XLSX, index=False)
    print(f'Excel salvo: {OUTPUT_XLSX}')

    # mostrar as primeiras linhas
    print('\nPrimeiras 10 linhas:')
    print(df.head(10).to_string(index=False))

if __name__ == '__main__':
    main()


Exportando coleção "patrimonios_santos" para DataFrame...
Número de documentos: 32
CSV salvo: patrimonios_santos_export.csv


ValueError: Excel does not support datetimes with timezones. Please ensure that datetimes are timezone unaware before writing to Excel.

In [2]:
import pandas as pd

In [2]:
firebase = pd.read_csv('patrimonios_santos_export.csv')

In [12]:
firebase

,_doc_id,nm_autor_projeto,ic_inauguracao,ds_fonte,nm_patrimonio,ds_funcionamento,nu_longitude,nu_latitude,ic_tombamento,ds_uso_original,ds_uso_atual,ds_localizacao,ds_historico,ds_endereco,ds_grau,ic_projeto,imageUrls
0,6A4BYFRKJ6F2fVTMCJph,Desconhecido,1991,https://memorialsantos.com.br/historia/,NECROPOLE ECUMENICA,Dias: Segunda a Domingo Horário: 24 horas Cust...,-46.343459,-23.948325,-,Cemitério/Mausoléu,Cemitério/Mausoléu,"Av. Dr. Nilo Peçanha, 50 - Marapé, Santos - SP...",Símbolo de tranquilidade e excelência nos serv...,"Velório do Memorial Necrópole Ecumênica, 50, A...",-,1983,NaN
1,7RxJEue6YG0SBl44V1hu,João Bernils,1924,http://condephaat.sp.gov.br/benstombados/teatr...,TEATRO COLISEU,Dias: No momento o teatro está fechado para r...,-51.160185,-23.322499,1983,Teatro - cultural,Teatro - cultural,"Rua Amador Bueno nº 237, Centro Histórico",Em 18 de julho de 1897 a Companhia Coliseu Sa...,"Rua Amador Bueno, Ipiranga, Higienópolis, Lond...",Condepasa e Condephaat,1909,https://res.cloudinary.com/dglfvvzg1/image/upl...
2,CdDBZovnsM0ur1bjn6hj,Desconhecido,1908,https://www.resjeroteirosbaixadasantista.prceu...,MUSEU DA PESCA,Dias: Fechado temporariamente para reforma H...,-46.307910,-23.986236,1998,Escola de aprendizes - marinheiros,Museu,"Av. Bartholomeu de Gusmão, 192 - Ponta da Prai...",O local onde se situa o prédio do Museu da Pe...,"Avenida Bartholomeu de Gusmão, Ponta da Praia,...",CONDEPHAAT,Desconhecido,NaN
3,Dd4WTvBZiDCZykQFsMuj,João Éboli,Entre 1880 e 1884,-,OUTEIRO DE SANTA CATARINA,Dias: Segunda `a Sexta Horário: 09h00 às 17h00...,-46.327340,-23.932876,1986-04-09 00:00:00+00:00,Capela,Sede administrativa da Fundação Arquivo e Memó...,"Rua Visconde do Rio Branco, 48 - Centro, Santo...",Construção: Após o outeiro ter passado por div...,"Rua Visconde do Rio Branco, Centro, Santos, Re...",NP‑1,Entre 1880 e 1884,https://res.cloudinary.com/dglfvvzg1/image/upl...
4,F7vmtLH41RBzI28pRCQ7,Engenheiro Plínio Botelho do Amaral,1939,https://www.camarasantos.sp.gov.br/um-pouco-de...,PREFEITURA MUNICIPAL DE SANTOS,Dias: segunda a sexta-feira Horário: das 8:00...,-46.328962,-23.933880,1985,Sede da Prefeitura e Câmara Municipal,Sede da Prefeitura e Câmara Municipal,"Praça Visc. de Mauá - Centro, Santos - SP, 110...","A construção reflete, através da linguagem arq...","Rua General Câmara, Centro, Santos, Região Ime...",Condephasa,1936,NaN
5,GLwPh9dIWoUaQ8GKI0p6,Desconhecido,1925,https://www.diariodolitoral.com.br/cotidiano/u...,EDUCANDÁRIO ANÁLIA FRANCO,Dias: Não tem Horário: Não tem Custo de ingres...,-46.332018,-23.960094,CONDEPASA (zona envoltória),Educandário,Abamdonado,"Av. Ana Costa, 285 - Gonzaga, Santos - SP, 110...",Contribuidor: Loja Maçonica Fraternidade de Sa...,"Avenida Anna Costa, Gonzaga, Santos, Região Im...",Zona de Proteção,Não consta,NaN
6,HT0uo43UC0yJh5q2hZ0r,Maximilian Emil Hehl,1924,https://www.diocesedesantos.com.br/paroquias/s...,CATEDRAL DE SANTOS,Dias: Segunda á Sábado Horário: 07:00 ás 18:...,-46.324618,-23.937143,2014,Igreja,Catedral,"Praça Patriarca José Bonifácio, S/N - Centro, ...","A Catedral de Santos, na Praça José Bonifácio,...","Praça Patriarca José Bonifácio, Centro, Santos...",CONDEPASA,1909,https://res.cloudinary.com/dglfvvzg1/image/upl...
7,IXwIOhpyPStCDvBzUHC2,Desconhecido,2022,https://www.santos.sp.gov.br/?q=noticia/homena...,MEMORIAL JOSÉ BONIFÁCIO,Dias: Segunda à Segunda Horário: 8h às 17h Cus...,-46.328962,-23.933880,-,Memorial,Memorial,"Praça Mauá De Santos - Centro, Santos - SP, 11...",Benfeitor: Bandeirantes Deicmar\nEm 7 de setem...,"Rua General Câmara, Centro, Santos, Região Ime...",-,NaN,NaN
8,JLz4jGGaRcpEKMDnxdpz,Iniciativa do prefeito Antonio Gomide Ribeiro ...,1945,https://www.vivasantos.com.br/aquario/historia...,AQUARIO DE SANTOS,Dias: Terça a Domingo Horário: 9h às 18h. Cust...,-46.308375,-23.986274,CONDEPASA (análise),Aquário,Aquário,"Praça Luiz La Scala, s/no - Ponta da Praia, Sa...","O Aquário Municipal de Santos, o mais antigo d...","Aquário Municipal de Santos, Avenida Ba

In [11]:
firebase["imageUrls"].isnull().sum()

np.int64(14)

In [13]:
firebase[firebase["imageUrls"].isnull()]

,_doc_id,nm_autor_projeto,ic_inauguracao,ds_fonte,nm_patrimonio,ds_funcionamento,nu_longitude,nu_latitude,ic_tombamento,ds_uso_original,ds_uso_atual,ds_localizacao,ds_historico,ds_endereco,ds_grau,ic_projeto,imageUrls
0,6A4BYFRKJ6F2fVTMCJph,Desconhecido,1991,https://memorialsantos.com.br/historia/,NECROPOLE ECUMENICA,Dias: Segunda a Domingo Horário: 24 horas Cust...,-46.343459,-23.948325,-,Cemitério/Mausoléu,Cemitério/Mausoléu,"Av. Dr. Nilo Peçanha, 50 - Marapé, Santos - SP...",Símbolo de tranquilidade e excelência nos serv...,"Velório do Memorial Necrópole Ecumênica, 50, A...",-,1983,NaN
2,CdDBZovnsM0ur1bjn6hj,Desconhecido,1908,https://www.resjeroteirosbaixadasantista.prceu...,MUSEU DA PESCA,Dias: Fechado temporariamente para reforma H...,-46.307910,-23.986236,1998,Escola de aprendizes - marinheiros,Museu,"Av. Bartholomeu de Gusmão, 192 - Ponta da Prai...",O local onde se situa o prédio do Museu da Pe...,"Avenida Bartholomeu de Gusmão, Ponta da Praia,...",CONDEPHAAT,Desconhecido,NaN
4,F7vmtLH41RBzI28pRCQ7,Engenheiro Plínio Botelho do Amaral,1939,https://www.camarasantos.sp.gov.br/um-pouco-de...,PREFEITURA MUNICIPAL DE SANTOS,Dias: segunda a sexta-feira Horário: das 8:00...,-46.328962,-23.933880,1985,Sede da Prefeitura e Câmara Municipal,Sede da Prefeitura e Câmara Municipal,"Praça Visc. de Mauá - Centro, Santos - SP, 110...","A construção reflete, através da linguagem arq...","Rua General Câmara, Centro, Santos, Região Ime...",Condephasa,1936,NaN
5,GLwPh9dIWoUaQ8GKI0p6,Desconhecido,1925,https://www.diariodolitoral.com.br/cotidiano/u...,EDUCANDÁRIO ANÁLIA FRANCO,Dias: Não tem Horário: Não tem Custo de ingres...,-46.332018,-23.960094,CONDEPASA (zona envoltória),Educandário,Abamdonado,"Av. Ana Costa, 285 - Gonzaga, Santos - SP, 110...",Contribuidor: Loja Maçonica Fraternidade de Sa...,"Avenida Anna Costa, Gonzaga, Santos, Região Im...",Zona de Proteção,Não consta,NaN
7,IXwIOhpyPStCDvBzUHC2,Desconhecido,2022,https://www.santos.sp.gov.br/?q=noticia/homena...,MEMORIAL JOSÉ BONIFÁCIO,Dias: Segunda à Segunda Horário: 8h às 17h Cus...,-46.328962,-23.933880,-,Memorial,Memorial,"Praça Mauá De Santos - Centro, Santos - SP, 11...",Benfeitor: Bandeirantes Deicmar\nEm 7 de setem...,"Rua General Câmara, Centro, Santos, Região Ime...",-,NaN,NaN
8,JLz4jGGaRcpEKMDnxdpz,Iniciativa do prefeito Antonio Gomide Ribeiro ...,1945,https://www.vivasantos.com.br/aquario/historia...,AQUARIO DE SANTOS,Dias: Terça a Domingo Horário: 9h às 18h. Cust...,-46.308375,-23.986274,CONDEPASA (análise),Aquário,Aquário,"Praça Luiz La Scala, s/no - Ponta da Praia, Sa...","O Aquário Municipal de Santos, o mais antigo d...","Aquário Municipal de Santos, Avenida Bartholom...",-,NaN,NaN
11,N5N40vqlncWj2CymQW9h,Fundada por Frei Manoel de Santa Maria,1640,https://hoteisemsantos.wordpress.com/pontos-t...,IGREJA DO VALONGO,Dias: Terça a Domingo Horário: 09:00 as 18:0...,-46.333903,-23.931326,1995 e 2003,Convento,Igreja,"Largo Marquês de Monte Alegre, 13 - Valongo, ...",A pedra fundamental do santuário foi assentada...,"Igreja do Valongo, 13, Rua São Bento, Valongo,...",CONDEPHAAT e IPHAN,Sem informações,NaN
13,QrUJhitzZr9dANTYB83z,Engenheiro Saturnino de Brito,1945,https://www.turismosantos.com.br/?q=pt-br/cont...,ORQUIDÁRIO,"Dias: Ter a dom Horário: , das 9h às 18h. Bi...",-46.348881,-23.965350,Sem informações,Parque indígena,Parque Orquidário,"Praça Washington, s/n - José Menino, Santos - ...",O Orquidário Municipal foi inaugurado em 11 d...,"Praça Washington, Pompéia, José Menino, Santos...",Sem informações,1903,NaN
14,RBnigEtjTtFoPYiI9LbU,Manoel Joaquim Ferreira Metto,1865,https://www.fundasantos.org.br/page.php?78\nht...,CASA DA FRONTARIA,Dias: visitação através de agendamento prévio ...,-46.331891,-23.932544,1973,Residência,"1868 foi armazém e escritório, em 1940 foi hot...","Rua do Comércio, 92 - Centro, Santos - SP, 110...","Sete mil azulejos em alto-relevo, importados ...","Rua do Comércio, Valongo, Centro, Santos, Regi...","IPHAN, Condephaat e Condepasa",1865,NaN
15,

In [3]:
firebase[firebase["imageUrls"].isnull()]

,_doc_id,nm_autor_projeto,imageUrls,ic_inauguracao,ds_fonte,nm_patrimonio,ds_funcionamento,nu_longitude,nu_latitude,ic_tombamento,ds_uso_original,ds_uso_atual,ds_localizacao,ds_historico,ds_endereco,ds_grau,ic_projeto
2,CdDBZovnsM0ur1bjn6hj,Desconhecido,NaN,1908,https://www.resjeroteirosbaixadasantista.prceu...,MUSEU DA PESCA,Dias: Fechado temporariamente para reforma H...,-46.307910,-23.986236,1998,Escola de aprendizes - marinheiros,Museu,"Av. Bartholomeu de Gusmão, 192 - Ponta da Prai...",O local onde se situa o prédio do Museu da Pe...,"Avenida Bartholomeu de Gusmão, Ponta da Praia,...",CONDEPHAAT,Desconhecido
4,F7vmtLH41RBzI28pRCQ7,Engenheiro Plínio Botelho do Amaral,NaN,1939,https://www.camarasantos.sp.gov.br/um-pouco-de...,PREFEITURA MUNICIPAL DE SANTOS,Dias: segunda a sexta-feira Horário: das 8:00...,-46.328962,-23.933880,1985,Sede da Prefeitura e Câmara Municipal,Sede da Prefeitura e Câmara Municipal,"Praça Visc. de Mauá - Centro, Santos - SP, 110...","A construção reflete, através da linguagem arq...","Rua General Câmara, Centro, Santos, Região Ime...",Condephasa,1936
5,GLwPh9dIWoUaQ8GKI0p6,Desconhecido,NaN,1925,https://www.diariodolitoral.com.br/cotidiano/u...,EDUCANDÁRIO ANÁLIA FRANCO,Dias: Não tem Horário: Não tem Custo de ingres...,-46.332018,-23.960094,CONDEPASA (zona envoltória),Educandário,Abamdonado,"Av. Ana Costa, 285 - Gonzaga, Santos - SP, 110...",Contribuidor: Loja Maçonica Fraternidade de Sa...,"Avenida Anna Costa, Gonzaga, Santos, Região Im...",Zona de Proteção,Não consta
8,JLz4jGGaRcpEKMDnxdpz,Iniciativa do prefeito Antonio Gomide Ribeiro ...,NaN,1945,https://www.vivasantos.com.br/aquario/historia...,AQUARIO DE SANTOS,Dias: Terça a Domingo Horário: 9h às 18h. Cust...,-46.308375,-23.986274,CONDEPASA (análise),Aquário,Aquário,"Praça Luiz La Scala, s/no - Ponta da Praia, Sa...","O Aquário Municipal de Santos, o mais antigo d...","Aquário Municipal de Santos, Avenida Bartholom...",-,NaN
11,N5N40vqlncWj2CymQW9h,Fundada por Frei Manoel de Santa Maria,NaN,1640,https://hoteisemsantos.wordpress.com/pontos-t...,IGREJA DO VALONGO,Dias: Terça a Domingo Horário: 09:00 as 18:0...,-46.333903,-23.931326,1995 e 2003,Convento,Igreja,"Largo Marquês de Monte Alegre, 13 - Valongo, ...",A pedra fundamental do santuário foi assentada...,"Igreja do Valongo, 13, Rua São Bento, Valongo,...",CONDEPHAAT e IPHAN,Sem informações
13,QrUJhitzZr9dANTYB83z,Engenheiro Saturnino de Brito,NaN,1945,https://www.turismosantos.com.br/?q=pt-br/cont...,ORQUIDÁRIO,"Dias: Ter a dom Horário: , das 9h às 18h. Bi...",-46.348881,-23.965350,Sem informações,Parque indígena,Parque Orquidário,"Praça Washington, s/n - José Menino, Santos - ...",O Orquidário Municipal foi inaugurado em 11 d...,"Praça Washington, Pompéia, José Menino, Santos...",Sem informações,1903
16,WmNHu7OJtWahJA8Arqxc,Garcia Dias de Ávila\n,NaN,1611,https://www.turismosantos.com.br/?q=pt-br/cont...,SANTUÁRIO DE NOSSA SERNHORA DO MONTE SERRAT,Dias e Horários: - Feirinha de Artesanato: to...,-46.327823,-23.938945,1993,Religioso,Religioso,"Caminho Monsenhor Moreira, 33, Santos, SP","São quatro minutos de pura emoção, subindo de ...","Caminho Monsenhor Moreira, Vila Monte Serrat, ...",CONDEPASA,1599
21,f2Xr21Un46udNqgAxpp6,"Osvaldo Correa Gonçalves, Abrahão Sanoviks, Jú...",NaN,1972,https://www.resjeroteirosbaixadasantista.prceu...,CENTRO CULTURAL PATRÍCIA GALVÃO,Dias: Segunda a sexta Horário: 9h às 18h Cu...,-46.332323,-23.944863,"1959, 1981 e 1990",Centro Cultural,Centro Cultural,"Av. Senador Pinheiro Machado, 48 - Vila Matias...",A fundação do centro cultural se deve à escri...,"Avenida Senador Pinheiro Machado, Vila Matias,...","IBPC, CONDEPHAAT E CONDEPASA",1970
